# 07 — Soft Voting Ensemble
Kết hợp Decision Tree, Naive Bayes, Bagging, Random Forest và AdaBoost bằng trung bình xác suất.

In [ ]:
from pathlib import Path
import json, hashlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform, sklearn
import importlib
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    average_precision_score, confusion_matrix, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.dummy import DummyClassifier
from securaai_ml.data import load_training_data
import securaai_ml.modeling as modeling_module
importlib.reload(modeling_module)  # tránh module cache cũ khi source vừa được cập nhật
from securaai_ml.modeling import (candidate_factories, decision_scores, parameter_grids,
    teaching_candidate_factories, teaching_parameter_grids)
from securaai_ml.schema import MODEL_FEATURES, TARGET_COLUMN, GROUP_COLUMN
from securaai_ml.train import temporal_train_validation_test

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data, audit = load_training_data(ROOT / 'data' / 'raw')
train, validation, test = temporal_train_validation_test(data)
X_train, y_train = train.loc[:, MODEL_FEATURES], train[TARGET_COLUMN]
X_validation, y_validation = validation.loc[:, MODEL_FEATURES], validation[TARGET_COLUMN]
groups = train[GROUP_COLUMN]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
audit
from sklearn.ensemble import VotingClassifier

## 1. Tạo ensemble
Linear SVM không có `predict_proba` nguyên bản nên không tham gia Soft Voting.

In [ ]:
factories=teaching_candidate_factories(42)
grids=teaching_parameter_grids()
base=['decision_tree','naive_bayes','bagging','random_forest','adaboost']
estimators=[]
for name in base:
    search=GridSearchCV(factories[name](),grids[name],scoring=make_scorer(fbeta_score,beta=2),cv=cv,n_jobs=1)
    search.fit(X_train,y_train,groups=groups)
    print(name, search.best_params_, search.best_score_)
    estimators.append((name, search.best_estimator_))
model=VotingClassifier(estimators=estimators,voting='soft',n_jobs=1)
model.fit(X_train,y_train)

## 2. Đánh giá Validation

In [ ]:
pred=model.predict(X_validation)
score=model.predict_proba(X_validation)[:,1]
ap=average_precision_score(y_validation,score)
result=pd.Series({'accuracy':accuracy_score(y_validation,pred),'precision':precision_score(y_validation,pred),'recall':recall_score(y_validation,pred),'f1':f1_score(y_validation,pred),'f2':fbeta_score(y_validation,pred,beta=2),'roc_auc':roc_auc_score(y_validation,score),'average_precision':ap})
display(result.to_frame('Validation'))
tn,fp,fn,tp=confusion_matrix(y_validation,pred,labels=[0,1]).ravel()
fig,axes=plt.subplots(1,3,figsize=(17,4.8))
ConfusionMatrixDisplay.from_predictions(y_validation,pred,display_labels=['BENIGN','MALICIOUS_APT'],ax=axes[0]); axes[0].set_title('Validation — Confusion Matrix')
RocCurveDisplay.from_predictions(y_validation,score,name=f"AUC={result['roc_auc']:.3f}",ax=axes[1]); axes[1].plot([0,1],[0,1],'--',color='gray'); axes[1].scatter([fp/(fp+tn)],[tp/(tp+fn)],color='darkorange',zorder=5,label='Decision threshold'); axes[1].set_title('ROC — VALIDATION'); axes[1].legend()
PrecisionRecallDisplay.from_predictions(y_validation,score,name=f'AP={ap:.3f}',ax=axes[2]); axes[2].axhline(float(y_validation.mean()),ls='--',color='gray',label=f'Positive rate={y_validation.mean():.3f}'); axes[2].set_title('Precision–Recall — VALIDATION'); axes[2].legend()
plt.tight_layout(); plt.show()

## 3. Lưu ý
Voting không mặc nhiên tốt hơn mô hình đơn. Chỉ chọn nếu Validation F2 thực sự cao hơn.